# SAAP – Stage 1: Smartphone Analytics

**Submitted by:** Tanisha Sharma  
**Programme:** Master of Computer Applications (Artificial Intelligence)  
**University:** Jain (Deemed-to-be) University, Bengaluru  
**Project:** Smartphone Analytics (SAAP)  
**Stage:** 1 – Database Setup, Data Quality Checks and Exploratory Data Analysis  
**Database name:** `saap_smartphone`

---

## Project overview

This project studies three linked datasets: smartphone specifications, user profiles and user ratings. The workflow follows the Stage 1 requirements: create a local MySQL database, establish a secure Python connection through SQLAlchemy, load all three datasets, retrieve analytical data through SQL, and perform Univariate, Bivariate and Multivariate EDA.

## Rubric coverage map

| Assessment area | Marks | What is demonstrated in this notebook |
|---|---:|---|
| Environment and database setup | 5 | Local MySQL connection, database creation, SQLAlchemy connection |
| Data preparation and quality | 5 | Dataset descriptions, dimensions, data types, NULL checks, duplicate checks, numeric ranges and IQR outlier flags |
| Database loading | Included in setup | All 3 source datasets loaded into MySQL and row counts verified |
| Univariate EDA | 10 | U1–U5: phone price, salary, internal memory, weight and battery size |
| Bivariate EDA | 10 | B1–B5: age/price, camera/price, performance/price, OS ratings and release date/price |
| Multivariate EDA | 10 | M1–M5: regional brand preference, occupation/price, model activity by brand, highest-rated model by brand, regional brand/price/salary |
| **Visible rubric total** | **40** | Each analysis includes a result table or chart and a written interpretation |

## Important methodological note

The supplied ratings dataset records **user ratings**, not verified sales transactions. Wherever the assignment uses wording such as “purchase behaviour” or “most purchased”, this notebook uses the number of rating records as a clearly labelled **rating-activity proxy**. It does not claim that a rating proves a purchase.

## Before final submission

This is a self-contained `.ipynb` file: the three supplied CSV datasets are embedded in the notebook, so no separate CSV download is needed. Run it in **local Jupyter on the same Windows computer as MySQL Server**, execute every cell from top to bottom, verify the generated tables/charts, and save the executed notebook. The notebook has not been run against the user's local MySQL service from this environment, so the local connection and final execution must still be verified on that computer.


## 1. Environment setup

Run the installation cell once if any package is missing. If Jupyter asks to restart the kernel after installation, restart it and then run the import cell again.

In [1]:
%pip install pandas numpy matplotlib sqlalchemy pymysql openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
from pathlib import Path
from getpass import getpass
from urllib.parse import quote_plus

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["figure.figsize"] = (10, 5)

def correlation_summary(value):
    magnitude = abs(value)
    if magnitude < 0.20:
        strength = "very weak"
    elif magnitude < 0.40:
        strength = "weak"
    elif magnitude < 0.60:
        strength = "moderate"
    elif magnitude < 0.80:
        strength = "strong"
    else:
        strength = "very strong"
    direction = "positive" if value > 0 else ("negative" if value < 0 else "no")
    return f"The relationship is {strength} and {direction} (correlation = {value:.3f})."

print("Libraries imported successfully.")

Libraries imported successfully.


### 1.1 Built-in source datasets

To keep this submission in one notebook file, the original three CSV datasets are embedded as a compressed data bundle. The source records are read without adding, deleting or changing observations. The embedded files are the supplied Stage 1 datasets.


In [3]:
# Decode the three original CSV files embedded in this notebook.
# This makes the submission self-contained; no separate CSV files are required.
import base64
import zipfile
from io import BytesIO, TextIOWrapper

DATA_BUNDLE_B64 = """UEsDBBQAAAAIACCWbV1Fw64nsQQAAKALAAAYAAAAc2FhcF9jZWxscGhvbmVzX2RhdGEuY3N2hZZNbuNGEIX3PgWXY6BMdVX/MjsvYiML24I9CYTZBByp7SFAkQIlZ6ycJvfIIkAulCukukmKP1IQQQZkQfz4uurVK/7z519rX5a7b3Xlfy028LXJqw1s640vod75Jj8U1VuyP+4PfgtFdfBNlZfJ1m/r5gjPtw/Av3mtm21erT1s86JK1vmWL4O9L18L3//3NT/wpcdkX/zuYb9uvK/az9998fbtALumWPtPPz0+X0PjS5/vfbLJD/5KwO1uV3oolkFg8vJj8okE0TUUTy+A5ECBTUnyR7BAAvmL1AIqBdIqK1NUWf9SgG4h5CJcf4VTLsrkoaiKCdVSoPKblHSgU75eIRgU2qZmjCW1EFnA4hXNsVOi7oiSlACTIqBV/0OUZ0KXTR2ppA0YpoaDtVRhdKSSUOAsGUr5ixFLXWIlD/nHiOdSgR1PSU3M47qyWhRczjlQT4Gr5wgyXHxuA1HXlUxRe9hwMmFQpQbILFAsQsOuDNzu3/fwxVevAeLgtto0NVsxlM0FASYggyAhBDciAzQZaJJEs7KxphukG6ETIX6I7ysLL/l2/169wX1e5h/H5Jab0t9BctEAU2lAC76VDnzD6jDjP6k08r1InkzjzlgM6FkskRWkrNLxtT2LyyW461a6iVYHRAuBbRWzM6yWkyIYkKnNYl1pAKNjcZY0m/zEzWxsT29yMSe/EM3K61KH4fT8W2lF50pjQRkkwXc9vbhneiGoI+MFcrIsuY9TvA02aPFKn6qrwTpNNrppBKVL0J/LA+fHlJqlhosseKpEX49QUb6d5LFPxYQq59QvyV1Z7BI5g7KxTHQ+V0JGJgeJkyOp0V54I9xgL1QX6HW5GdHDXGGYABnHn23CPubCcEEt11pbVPSfeA33df3GI7YsPnyZmOTvPy7NR7Qvt4zTJA1zYMPwunmquHbm2HJoZtx8ZjguqAtqmaqwzyoHMiN2xSRUCRfCdlQ7V8v5MhMcIoENGz0RjCdkGzHhJheCcCTZwVPlo8Ue62aTPJI4GxIXg8L0VlNh8DhqEO1s+HgRqM4e2RRLn8/qy2nKann0BmwWPMJyY9+4ALqFsaQehiIG9dmwmZAPwxjHpQXGoZTBtgyJFsDBAoQj5lwcYipiKQP2BGS9xmTzPgW2MBN78Ww+7XY13BXVJlnpieLOtsgY6s4/aOZmOEOc5Io9PQQOL+Jfit9qWDlxxgpT20ZNqKQ9kTgLDOc5Bydlp66QglWR19sCnv1mWySP9YH3FU5OH9JWxdYMcRs6zutBY+ZCdMcjZzeCRkfWPRnx8wWRNtVZjBY0wz4IJZWchZxm3ZwKfcMdHqjmRKULfeeT6/7kZtwmmfFeoHHISlwgdQ8Ttocu63Wd3KkzZ2bR8DTLVsnbRs+i23ZDyqXlLVZXR1jxg1uRT8Tqdu/z7FM3+6clRmGJKU06lRHWrS7K4KE+1E1d5vFDcp+8HI7s1f4xbTqhlMrOrUNqswGUNFYH+9uTAaQ4Ay/L/BixeD1f4PyAETwwXrrsBOmcCwMaDNrplXiOfW/8nCc6nhp2bZg1Mi7FyOtDiX9/xqu/+2Z+/PhwgGEDjB80gk6+ndEiG89q+3DQra9/AVBLAwQUAAAACAAhlm1dNzgu7fwGAADeEgAAGAAAAHNhYXBfY2VsbHBob25lX3VzZXJzLmNzdpVYy3LbOhLd6yu0nKnCggTfS0ZWYiW27CvpJnWzSUEULCImQRdIOlfza7OYT5pfmAO+BMqR5PEizqIBdp/uPufA//33f+qSqx9iR5rfkuWcrPheFPIfM1Ed/knWLGPq8EPIH4vlirA9J3sud1yRIknqF1YhcmKRFcvJDc9SQbzA8iyLOCH5yHOWcXLDKjZlkmWHsprYZF0r9pPMUi4lQ7Rv4Ye4Vh+94SyfZpzhCxMfV/It6y8Oqb7Y8/rQhXwqVN5kMK14ksoiK/aHSUhiVcvukO8E+hAdDt0jkz3uti0SbxHwgck9ywrFiev5FHHUQQwCfxXqWcdRsk5VXQnypcieUQpxXapTpkOBcZIUtayYRHk+WedKIDpOc75jW7ZDvN8AYvfxJf4pJ9QFaGmdkcdachTnNcUFV4pDgt81JEbaboM3DX6HIPVJnPG/uejgeJP7x8UyXs7mExx/VBx5941xndDVkdEQKSSTCZ/gbCx/sswEJGoKdPrQdVtgRG6ZOt7oR7YO8wccHl7Kad8OxwLMv3jVN9sNHABCHLdtxmK2mT48PYlEh1LUJNjwedzrAhYnakP/iKfr4qn6xRQ/3u6Qr0wPRYO163oaBGeYudt5fLe5ncWr+cTxyBq7MJQWul7gEddu7y559jTl+UtWHPhu4mhwxb/YEbPQR4nEpW30lslnIfcTJyB/5sd8Ha8ZeZTWfV7WquQTLMxfTA31j6+a/80TzODrsSSPYrHkgdwe0Od2zrwIw65Hvc21B2HHX3lWvOgzDvlcpPJYHXa1XdXmRHxzv1gu1ptVvFl8nU8fPn5czOarieeTzQGJ3df5VrcxaNDzBvTyLiPfIl+EYrIP9CyNMyrpWvj7gQ5cshQlM/bF90d19AUHEZnvuDQm36cNkKeBoUW+sVLkXbMt3wzqtjp0yR/1FkUZ8AV2hM863fp/A3RpUZe6gaFHHnIwnhH8+5GPbPJd93AAeDzF3+Yfpjfz9eLTchJ5aN8rJs1YZN+12918g9fmiFcU4ls5U1VqQIY2jlmuOSASlk3nci8kR8lRhIPqlZdpP2J+QL1mDS59T6f0lQmcGvbYavbYda5QlW1hwplKWb92gaUZxaEXp822PE1ZPX+7p/y9horwfBrvciFFWSlWFZrMsYrPZcoO5hyFo+/1Y2qD+ZdM7tIRiQZ+oGP9fpQwJZVuvW3b5I49G+W7XsPUzsC385s/Z6jhYYlgijAlSuNm75Rs1thkKATwsR3yifMqrY2kOz3EznXhIGUmMgS7oN09z3pcIupqfugSrtp+CyYRCQAx2ikUudtEx2oZZ0g5LXL8p5E3KNZfdVk/jTZhBFzFWZI2oZHmu4R3itVe6g4C8aJ4KbCh1RRdkeVLoap2JpIifwFVQUot8qHYGl86yyU2VHiWMpUJPvDOmXZCR+ev7EhqXhSdu9MnKzHw2BvdPLcAWMh18ZKKnpovJB2Ruzph5XFWTiXv3DfQGsycIWpo7lh+z52Ec5rnhrrAbPiaPaPLWw1heVTiwN6xL65uRVZws2/nQrWn+akppt34d+mL7Xrkk2LJ1U5DSdYie2VGq0+c1dhqQiKXBTN58sIyei6oO1WGmI+34BzJYVNvWVliAE0lDpu8ouOEgLQUlHivOIdJtPQidRj5YajdS+cwkkKC1OqksdW2T8lnljybyNuWQ5vhG2me7QP5YivGWTiWnoTuZma6VGyy9vBlMfjkgDq26dqHm4MQN5el4Qyv+gw7dNps9P1HLfRsXWh3zHSpdgSBLLReGZsz4rczowMDAEdT8hezyWBQ/7rvoJYDDEBL5gp4w8BeceGQtnuRpAx8bJqCk+k6m3VIvrDXg1Htu5aWQotuGQ6Mqj2Vo3V8N19P7+Nl/El/C2pwA1kwnY7vOw1z0/7IY61QS3k0lxRUuYFIgMr6d8P7yJIi4k5kh5G0vmePKFCbge8LUzthgqnvXydBqrcHHsdgBogrxse7zDwUST2ySgmsWG9ao8Ccu18gB40HjN53hgfQdlCPC2tL4fceMph11b83QqtFYND0vdCEgDcE1Q+OWgrk3KlqSFv9b58Q2oDyEo8pWJNbLtXhvGPdjFQXJwIS5zDk5lN0fGK+g1p1wSH5KhJ4KXMbYF6BfW8QB66krnZQFdOPsMEUXhlDDFu8w+Hj+27EBsYLjEJvPrNXvmsBeTOsw4y62hHJ47skGj3Vjl/29B8nDNt74cs+BAMQ9Dc2Sg++Gz7d3+nZcG6F2g+S5YajN+gMjqfWjG/4Mgo5wns441z+v2OKd9siV4OPPv2TybmtgKTNc72Mxsic2unfsDf1PIwaLDgbfe8C31NtV/FGYJnB34Gtu3GF0byArPiWJ8mIn2hktQJz2nIAtaq34tigkRK9fST/D1BLAwQUAAAACAAglm1dDK79ReEKAADaIAAAGwAAAHNhYXBfY2VsbHBob25lc19yYXRpbmdzLmNzdkVZW661Nw1971j2Q+LcR4MQ/IJKVUEFRscDQ2IK9bp4Vzo6XvElX+I4ieP9///+7z//+vHbn37+6+cvP3755Z9//8evP9D47c///vnXv/3UPqN9epL1Gfm/t89L8tiI8Ykkl/8jqNY3W/2xNdLgp/45n5v/R4Ik/X4WWhKN+OwkcamS1lRZbE02INmfPtJqo4+dJL9MZn5rp8IBxhc3+st+s5scZ+qdz4Qs+T/lMF8aX4z0JRkigW8ladnL/eQ883/vVOxHok5mcBac9QOFQ5JwxC3HSmafnBZmrnbIKEcgY3eSX0gan5g5wB7yKyi7C0x3g26RHOui+pE6Rwt+xyACY0q9Dcfh8xv9DlCsHWafXyab9tufz/ZGv/TuAX3qBmuYPQQGBYolQI/oHO2udjrygG6RbrVeZsN8Dir9DIcnWfmRJOmqAB3wOdo5Hbatdm2VY+ukW/yg/gbdoNLfHh19cCgfkqcJ5RGlgDUBveoIS5eOoU+wPiIdvkgaiELQI5pW5G+ppS9oPUJtrn86LP86aK4Q6TLfbMQJ6RHl7gmGqMyqm4X2g/4CRayj+dSUq19G+wbJsV5SNRlMaG5pr1x87GbE4GCsbtK0J11qBxw9sMmp3my1ZTWljIDEXsO4k2INRmit0Nxud7UZp4NxG2w38beaOY3se5DdAHKwD7QPCbZIWIy9TlC0fw2XAddjLE+O0RWgsUXTCYPtoTaCCeyLDy+tdlJ4LglWa2x/n9uGbROcFkmWSKfDGG+k3cbPzYMPMq42KLc22oz3BIiIcRSdkHfr40RBu1k+JM9VS5rGGPCFeILiVEvC6IW0mS70lnyJu622+ghMYXGZciwJsNUWl3WSfyzHeiRFMCdBn0mGSEwbYWBraH8mxYiS6DRIMMVmqCwsJcVHJNzk5oUV+dudctMPUgTUgu8fCI4PqqnZQ9p07aLvNyl8uRnTmxThmZRBsxH5C4R7eHtDbG6QSbXOO6ahn6ACguxM+QV0qd2PaTe/q40jJMloonHFHmF1DOs8bTnQbsrPPG2XpJhFEl6GaIbEA/fV4ZnBNhb1Nnr2erdfXutsY6ddzpV0dLF1Wl5cc1S/ZkNr6lgCpdKElfjXjOhSOKXQpKD78dI1ssAeutx4pAzvpPpC/qP+YtiQ7/Ys/rTesRwffl3zeM4oQPNzSUKERx2aCBC0n7SjrEoPC/aWRptUXnm+tN5SfILitng+N97ilfh4l03aL6mTIMtAL1ejuf6ac5B3dYw/3BVkP7WYKb2rMz8p1vM9TrwBMErBGMWIAl2S6fZV86u3DHT89tZ8ABAdJjJNgwJg4JDzDO5XeZtTRhoLEROc5qAH4NEEwEOeIKyj/K9x49C+Iosd/ME7pbbFWjonCC57d6CQw+QmwSsVfY4xJBBh0SmG/bBrtE47AJhnEMgzSCQkiV0SprpNVwMZYVXleV3Jc1eiuAmQYhDw4CdnmrPMGMjtelPQATx31qWJnHmSVvvQotd8gBCKANyZAOq9K20DYKZDDpcVV/2RORNOI62ectMnTudXo2aAa35K5XEcodyPKjQeym8ITJG2CMgfww+GPnRkAJwvp5vDfQ2gVe/IWQ+plrb7sCXgcnWdxmQ8g1kayzaKlD7LE+ujQS3vHSB9SDsftHfarI8WdukSJXhW4SFCFcVv9ysGYBYnmjn9C6LAMBhf812cbmD3vZoMtz1p16ogeVzS4JuFuRWXiRdPSKTd118N/tnVUbs/dMGAPgsUYtG85YMBa91eytEK8HUIEenwgRND9xjBMxiW3BKEGVEadDo0v73pYTUd1QCPT67p/RuzHl2+gAF0xMXUOd/51jFjuj9GfWwfInpWHAFNbXtGemh0vVg2gYJHTxIjLZOeKwBX+VTXm2ETXEuqvUuBEQaTUba3RGGAJDzBKz/gVrik2r/IvDWWVz57n2WJvflqsV755ele7shHIUlK76I9C2xLfIAjp+VklQQXK6weXIPhh1iffmknlYNnU2ZJDucwv/fWbF7d2bwPoW0dXS7zeznN8LwSyOkzHFEzHKgQXevo3E/wTMdXt3Ok4ZEOR0AC9ztq7KO6G3L/HA6KOZRkU/dVN6OAlHmJHgJF0JzK3wjColGgL4tGKfOlDlBU+2auj3yz/mD4S8sBP/FSkkr/6u5SYUjNOuEApHzr0xefDgKePmBsM1SLmM6OKCqgLTAvX7BkUILHBufGN4goE7BezxAA+mzxkRmSXAPmlJ3vGDO43ZcTeIDNbpnPbgKdEWs6D1iV0KzpI3tN76TlvB+Ai4P3iRjbN8ravlEAroGWdNUJwNeMOa8Yr6xV9+EhTBq86XerE8WvFYJj+mzDFdjNxaTtSh+N1V04Odi1DXbozNnh03mHprbD4bbpZ6ksazQLuAF3XaIJNBC8Io7otYDvNABWx/aseU7dMZsn9JKEYblfTfMpg9iPDxdQhch2vkvOKVFx+hfRxft5kKeC8DjvJmBknesFP9enxUHoSkWb+bjWCHC+ulrfW/fa5emwBbhdLp+7S5wCcu+tg+MOx90dLP5QAuM39dYHuMXAGB7DcQmgFkHAyH+TBV3QqE7o8CeHR2sqoRKghBh6BzwCxk80P4cpotPIYvmr+WkNwFpUa9pR0TwzAnU4vua8zxdB50eHinNoq99R3xwIkSHAYlibruDppbHE0SC4ZweBPjlVT6DuMoflvlZVTT1K0N11JbDdTzPtctL1jUIUZvG1RM79yrZZpaM6qlLvK9DM+TKw4wF4F5BTEhb7+FAqybSoC6zqd/HKAh0y8jMJgI7oy9VTBmmBY2PNyaluhArvoXxGgLXQP0qtsT0WZUVmqZvYOudDFVvZiR5XjuNoi0RVW1mctQDZAYA8rjqt0ejuptmIwxt+wgIc0/GVPHNUph2uh1CnWaRlQ9NG8h1+kZBRl3tHVxEOoKvf7pcT0SqksBxdtw3B4Uc7T9kYrvcDqOo8/MMAwbJodIOvLlcOElOmWaFqbUb5GKpCj6HrAkAHIJAK1oOvK1LuItZgp2zUvaq1MlZ3vBMnQZfTpsN1TJZKQW00VWukir7sMx2AYTC49TC6VcEzlvb/WDr8KGKNe1SMD5dkAJ4ZvYzjFFLooUj7CvUCHqGfTsHKbYm+AA8jWukj2yUIImsdXdOhpHYT6LcOVW+HRM2iUnFUnFrgU8uH4IfNrRneT1GdSOOW8f0o0q+SHkquAZMegC+DU5jNrpx+kxGM4vBMnU0PCoBXkl7Wx5xiqO2fuELJtTjqo2JZSbYkHDYYZfv4S5RLCZRobCoUgzINjcqSAfiwJ/APVqN+wRq+deao37b000IoMwfDbz4A5mgExxwb1S0x/YgkZxos65ZAJioDhPLoQdDlwuVJV3GR6FikkwFpuObvSiKBfLTLeZu/FYIeqm6Fy9yquhBca8iHuxZ66+kPoOBf3ffA8g+0AGGGjpfVbb3q/FquGJHTS1RWnOQKn5VLrydQnZWVfBOc0qlfDp3NhxJ+2emsXMpIw78cgPJtCnAtaBYMfsiJU6zhMTmlAvBPjMM30nIlIPQAmOJwINM1m1CWL5EdMusnTT/PyAmLdKOt78+bqlgLvQJfjo/l5TIm1Veh/dX6Cn324GXAz7ieACAH8vqVZPRS0QpvpzqrflJdLlPGqit6HfvsVDd1Oq2jbAiMZ8BtDtVjW7vFZ9065TCnyQD6nn+qJyiO0q919fMGOaesh63jy0FQ/A5QSwECFAMUAAAACAAglm1dRcOuJ7EEAACgCwAAGAAAAAAAAAAAAAAApIEAAAAAc2FhcF9jZWxscGhvbmVzX2RhdGEuY3N2UEsBAhQDFAAAAAgAIZZtXTc4Lu38BgAA3hIAABgAAAAAAAAAAAAAAKSB5wQAAHNhYXBfY2VsbHBob25lX3VzZXJzLmNzdlBLAQIUAxQAAAAIACCWbV0Mrv1F4QoAANogAAAbAAAAAAAAAAAAAACkgRkMAABzYWFwX2NlbGxwaG9uZXNfcmF0aW5ncy5jc3ZQSwUGAAAAAAMAAwDVAAAAMxcAAAAA"""

data_bundle = base64.b64decode(DATA_BUNDLE_B64)
with zipfile.ZipFile(BytesIO(data_bundle), mode="r") as dataset_zip:
    expected_files = {
        "cellphones_data": "saap_cellphones_data.csv",
        "cellphone_users": "saap_cellphone_users.csv",
        "cellphones_ratings": "saap_cellphones_ratings.csv",
    }
    loaded_sources = {}
    for table_name, csv_name in expected_files.items():
        with dataset_zip.open(csv_name) as raw_file:
            loaded_sources[table_name] = pd.read_csv(TextIOWrapper(raw_file, encoding="utf-8"))

phones_source = loaded_sources["cellphones_data"]
users_source = loaded_sources["cellphone_users"]
ratings_source = loaded_sources["cellphones_ratings"]

print("Embedded source datasets decoded successfully.")
for table_name, frame in loaded_sources.items():
    print(f"{table_name}: {frame.shape[0]} rows × {frame.shape[1]} columns")

expected_shapes = {
    "cellphones_data": (33, 14),
    "cellphone_users": (99, 7),
    "cellphones_ratings": (990, 3),
}
for table_name, expected_shape in expected_shapes.items():
    actual_shape = loaded_sources[table_name].shape
    assert actual_shape == expected_shape, (
        f"{table_name}: expected {expected_shape}, received {actual_shape}. "
        "Please confirm that the embedded source dataset is intact."
    )
print("All three source datasets match the expected dimensions.")


Embedded source datasets decoded successfully.
cellphones_data: 33 rows × 14 columns
cellphone_users: 99 rows × 7 columns
cellphones_ratings: 990 rows × 3 columns
All three source datasets match the expected dimensions.


## 2. Secure connection to local MySQL

The notebook connects to MySQL on the same computer using SQLAlchemy and PyMySQL. The password is requested securely with `getpass` and is not written into the notebook. The database `saap_smartphone` is created if it does not already exist.

**Requirement:** MySQL Server 8.0 or newer is needed for the window-function ranking queries used in the regional and model-level analyses. Start the MySQL service before running the connection cell.


In [ ]:
import platform
import socket

MYSQL_USER = "root"
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
DATABASE_NAME = "saap_smartphone"

print("Operating system running this notebook:", platform.system())
if platform.system() != "Windows":
    print(
        "WARNING: This notebook appears to be running outside Windows. "
        "If MySQL is installed on your Windows laptop, localhost here may refer "
        "to the remote notebook server instead of your laptop. Run this notebook "
        "in local Jupyter on the same laptop as MySQL."
    )

# Check whether a MySQL service is listening before requesting the password.
try:
    with socket.create_connection((MYSQL_HOST, MYSQL_PORT), timeout=3):
        print("A service is listening at localhost:3306.")
except OSError as error:
    raise ConnectionError(
        "Cannot reach MySQL at localhost:3306. Start the MySQL80 service in Windows "
        "Services, then run this cell again. Also make sure this notebook is running "
        "locally on the same computer as MySQL (not in Anaconda Cloud)."
    ) from error

MYSQL_PASSWORD = getpass("Enter your MySQL root password: ")
connection_url = (
    f"mysql+pymysql://{MYSQL_USER}:{quote_plus(MYSQL_PASSWORD)}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/"
)

try:
    server_engine = create_engine(connection_url, pool_pre_ping=True)
    with server_engine.begin() as connection:
        connection.execute(text(f"CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}`"))

    engine = create_engine(
        connection_url + DATABASE_NAME,
        pool_pre_ping=True
    )

    with engine.connect() as connection:
        current_database = connection.execute(text("SELECT DATABASE()")).scalar()
        mysql_version = connection.execute(text("SELECT VERSION()")).scalar()

    print("Connected database:", current_database)
    print("MySQL version:", mysql_version)
    assert current_database == DATABASE_NAME

    # ROW_NUMBER() used in regional/model rankings requires MySQL 8.0 or newer.
    version_numbers = tuple(int(part) for part in mysql_version.split("-")[0].split(".")[:2])
    if version_numbers < (8, 0):
        raise RuntimeError(
            "This notebook's ranking queries require MySQL 8.0 or newer. "
            "Please upgrade MySQL before continuing."
        )
except Exception:
    raise

Operating system running this notebook: Windows
A service is listening at localhost:3306.


## 3. Load and study the supplied datasets

The source files are read here only for initial inspection and database loading. All subsequent analytical datasets are retrieved from MySQL using SQL queries.

In [ ]:
# The original CSV records were decoded from the embedded bundle in Section 1.1.
phones_source = loaded_sources["cellphones_data"]
users_source = loaded_sources["cellphone_users"]
ratings_source = loaded_sources["cellphones_ratings"]

table_frames = {
    "cellphones_data": phones_source,
    "cellphone_users": users_source,
    "cellphones_ratings": ratings_source
}

expected_shapes = {
    "cellphones_data": (33, 14),
    "cellphone_users": (99, 7),
    "cellphones_ratings": (990, 3)
}

for table_name, frame in table_frames.items():
    print(f"{table_name}: {frame.shape[0]} rows × {frame.shape[1]} columns")
    assert frame.shape == expected_shapes[table_name], (
        f"Unexpected dimensions for {table_name}: {frame.shape}"
    )

print("\nPreview: cellphone models")
display(phones_source.head())
print("\nPreview: users")
display(users_source.head())
print("\nPreview: ratings")
display(ratings_source.head())

### 3.1 Dataset descriptions and column information

In [ ]:
source_datasets = table_frames

for table_name, frame in source_datasets.items():
    print(f"\n--- {table_name}: dataset description ---")
    print("Rows:", frame.shape[0], "| Columns:", frame.shape[1])
    description = pd.DataFrame({
        "column_name": frame.columns,
        "source_data_type": frame.dtypes.astype(str).values,
        "non_null_values": frame.notna().sum().values,
        "null_values": frame.isna().sum().values
    })
    display(description)

### 3.2 Load all three datasets into MySQL

The original column names are preserved so the SQL queries match the supplied data. `if_exists="replace"` creates/replaces only these project tables; it does not delete the database itself.

In [ ]:
table_frames = {
    "cellphones_data": phones_source,
    "cellphone_users": users_source,
    "cellphones_ratings": ratings_source
}

for table_name, frame in table_frames.items():
    frame.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=500,
        method="multi"
    )
    print(f"Loaded {len(frame)} rows into {table_name}")

### 3.3 Verify database tables and row counts

In [ ]:
with engine.connect() as connection:
    table_counts = {}
    for table_name in table_frames:
        table_counts[table_name] = connection.execute(
            text(f"SELECT COUNT(*) FROM `{table_name}`")
        ).scalar()

count_report = pd.DataFrame({
    "table": list(table_counts.keys()),
    "source_rows": [len(table_frames[name]) for name in table_counts],
    "rows_in_mysql": [table_counts[name] for name in table_counts]
})
display(count_report)

assert all(
    count_report["source_rows"] == count_report["rows_in_mysql"]
), "At least one table's MySQL row count does not match its source CSV."

## 4. Data preprocessing and quality checks

As instructed, this stage checks the data but does **not** remove outliers, fill nulls, change source data types, create primary/foreign keys, or normalize the schema.

### 4.1 SQL table structure / data description

In [ ]:
for table_name in table_frames:
    print(f"\nStructure of {table_name}")
    with engine.connect() as connection:
        structure = pd.read_sql(
            text(f"DESCRIBE `{table_name}`"), connection
        )
    display(structure)

### 4.2 Missing-value check (SQL)

In [ ]:
# Build one SQL NULL-count query per table using the original column names.
# This is a data-quality check only; no values are filled or changed.
null_reports = []

for table_name, frame in table_frames.items():
    null_expressions = []
    for column_name in frame.columns:
        safe_column = column_name.replace("`", "``")
        alias = f"null_count_{len(null_expressions) + 1}"
        null_expressions.append(f"SUM(`{safe_column}` IS NULL) AS `{alias}`")

    query = "SELECT " + ", ".join(null_expressions) + f" FROM `{table_name}`"
    with engine.connect() as connection:
        result = pd.read_sql(text(query), connection)

    for position, column_name in enumerate(frame.columns, start=1):
        null_reports.append({
            "table": table_name,
            "column": column_name,
            "null_count": int(result.iloc[0, position - 1] or 0)
        })

null_report = pd.DataFrame(null_reports)
display(null_report)
print("Total missing values across all three SQL tables:", int(null_report["null_count"].sum()))

### 4.3 Duplicate-row checks (SQL)

In [ ]:
# Check exact duplicate rows in each SQL table and repeated identifiers separately.
duplicate_report = []

for table_name, frame in table_frames.items():
    quoted_columns = ", ".join(
        f"`{column_name.replace('`', '``')}`" for column_name in frame.columns
    )
    full_duplicate_query = f"""
        SELECT COUNT(*) AS duplicate_groups
        FROM (
            SELECT {quoted_columns}, COUNT(*) AS row_count
            FROM `{table_name}`
            GROUP BY {quoted_columns}
            HAVING COUNT(*) > 1
        ) AS duplicate_rows
    """
    with engine.connect() as connection:
        full_duplicate_groups = pd.read_sql(
            text(full_duplicate_query), connection
        ).iloc[0, 0]

    duplicate_report.append({
        "table": table_name,
        "check": "Exact duplicate rows",
        "duplicate_groups": int(full_duplicate_groups)
    })

identifier_columns = {
    "cellphones_data": ["cellphone_id"],
    "cellphone_users": ["user_id"],
    "cellphones_ratings": ["user_id", "cellphone_id"]
}

for table_name, columns in identifier_columns.items():
    group_columns = ", ".join(f"`{column}`" for column in columns)
    duplicate_key_query = f"""
        SELECT COUNT(*) AS duplicate_groups
        FROM (
            SELECT {group_columns}, COUNT(*) AS n
            FROM `{table_name}`
            GROUP BY {group_columns}
            HAVING COUNT(*) > 1
        ) AS repeated_keys
    """
    with engine.connect() as connection:
        duplicate_key_groups = pd.read_sql(
            text(duplicate_key_query), connection
        ).iloc[0, 0]

    duplicate_report.append({
        "table": table_name,
        "check": "Repeated identifier combination",
        "duplicate_groups": int(duplicate_key_groups)
    })

duplicate_report = pd.DataFrame(duplicate_report)
display(duplicate_report)
print("No records are deleted or modified during these checks.")

### 4.4 Relationship integrity checks

The source data is checked for rating rows whose user ID or cellphone ID does not exist in the corresponding master table. This is a diagnostic check only; no records are removed or altered. The source tables are intentionally loaded without primary-key or foreign-key constraints, in line with the Stage 1 instructions.


In [ ]:
integrity_query = """
SELECT
    SUM(CASE WHEN u.user_id IS NULL THEN 1 ELSE 0 END) AS ratings_with_unknown_user,
    SUM(CASE WHEN p.cellphone_id IS NULL THEN 1 ELSE 0 END) AS ratings_with_unknown_phone
FROM cellphones_ratings r
LEFT JOIN cellphone_users u ON r.user_id = u.user_id
LEFT JOIN cellphones_data p ON r.cellphone_id = p.cellphone_id
"""
with engine.connect() as connection:
    integrity_report = pd.read_sql(text(integrity_query), connection)
display(integrity_report)
print("Interpretation: Zero in both columns means every rating row links to a known user and a known phone in the supplied master tables.")


### 4.5 Numeric range checks and possible outliers

The IQR method flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. This is a diagnostic check only. Flagged values are not deleted or modified.

In [ ]:
# Numerical ranges are retrieved from MySQL. Identifier columns are excluded.
numeric_columns = {
    "cellphones_data": [
        "internal memory", "RAM", "performance", "main camera",
        "selfie camera", "battery size", "screen size", "weight", "price(INR)"
    ],
    "cellphone_users": ["Salary_in_INR", "age"],
    "cellphones_ratings": ["rating"]
}

range_rows = []
for table_name, columns in numeric_columns.items():
    for column_name in columns:
        safe_column = column_name.replace("`", "``")
        query = f"""
            SELECT MIN(`{safe_column}`) AS minimum_value,
                   MAX(`{safe_column}`) AS maximum_value,
                   AVG(`{safe_column}`) AS average_value
            FROM `{table_name}`
        """
        with engine.connect() as connection:
            result = pd.read_sql(text(query), connection).iloc[0]

        range_rows.append({
            "table": table_name,
            "measure": column_name,
            "minimum": result["minimum_value"],
            "maximum": result["maximum_value"],
            "average": result["average_value"]
        })

numeric_ranges = pd.DataFrame(range_rows)
display(numeric_ranges)

In [ ]:
# IQR outlier detection is performed on SQL-retrieved values.
# It flags unusual observations but does NOT remove or alter them.
outlier_rows = []

for table_name, columns in numeric_columns.items():
    for column_name in columns:
        safe_column = column_name.replace("`", "``")
        query = (
            f"SELECT `{safe_column}` AS value FROM `{table_name}` "
            f"WHERE `{safe_column}` IS NOT NULL"
        )
        with engine.connect() as connection:
            values = pd.read_sql(text(query), connection)["value"].dropna()

        if len(values) == 0:
            continue

        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        flagged = int(((values < lower_bound) | (values > upper_bound)).sum())

        outlier_rows.append({
            "table": table_name,
            "measure": column_name,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "possible_outlier_count": flagged
        })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)
print(
    "Interpretation: IQR flags potential extreme observations for review. "
    "A flagged value is not automatically an error. No outliers are removed, "
    "because the assignment asks for checking rather than treatment."
)

## 5. Exploratory Data Analysis (EDA)

**Rule followed:** each analysis section first retrieves its data using a SQL query from MySQL. Python is then used only to visualize and interpret the SQL result.

# 5A. Univariate Analysis

### U1. Distribution of cellphone prices and price range

In [ ]:
q_u1 = """
SELECT `price(INR)` AS phone_price
FROM cellphones_data
WHERE `price(INR)` IS NOT NULL
"""
with engine.connect() as connection:
    u1 = pd.read_sql(text(q_u1), connection)

display(u1.describe())
plt.hist(u1["phone_price"], bins=10, edgecolor="black")
plt.title("Distribution of Cellphone Prices")
plt.xlabel("Price (INR)")
plt.ylabel("Number of phone models")
plt.show()

print(f"Price range: INR {u1.phone_price.min():,.2f} to INR {u1.phone_price.max():,.2f}.")
print(f"Mean price: INR {u1.phone_price.mean():,.2f}; median price: INR {u1.phone_price.median():,.2f}.")
print(f"Price distribution skewness: {u1.phone_price.skew():.3f}.")
print("Interpretation: Compare the mean and median to understand whether a few expensive models pull the average upward. The histogram shows where most listed models fall within the observed price range.")

### U2. User salary distribution and range

In [ ]:
q_u2 = """
SELECT `Salary_in_INR` AS salary
FROM cellphone_users
WHERE `Salary_in_INR` IS NOT NULL
"""
with engine.connect() as connection:
    u2 = pd.read_sql(text(q_u2), connection)

display(u2.describe())
plt.hist(u2["salary"], bins=12, edgecolor="black")
plt.title("Distribution of User Salaries")
plt.xlabel("Salary (INR)")
plt.ylabel("Number of users")
plt.show()

print(f"Salary range: INR {u2.salary.min():,.2f} to INR {u2.salary.max():,.2f}.")
print(f"Mean salary: INR {u2.salary.mean():,.2f}; median salary: INR {u2.salary.median():,.2f}.")
print(f"Salary distribution skewness: {u2.salary.skew():.3f}.")
print("Interpretation: The histogram shows the salary distribution among the 99 users. Comparing the mean and median helps identify whether the distribution is pulled toward higher or lower salaries.")

### U3. Distribution of internal memory sizes

In [ ]:
q_u3 = """
SELECT `internal memory` AS internal_memory, COUNT(*) AS model_count
FROM cellphones_data
GROUP BY `internal memory`
ORDER BY model_count DESC
"""
with engine.connect() as connection:
    u3 = pd.read_sql(text(q_u3), connection)

display(u3)
plt.bar(u3["internal_memory"].astype(str), u3["model_count"])
plt.title("Distribution of Internal Memory Sizes")
plt.xlabel("Internal memory")
plt.ylabel("Number of phone models")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

most_common_memory = u3.sort_values(["model_count", "internal_memory"], ascending=[False, True]).iloc[0]
print(f"Most common internal memory: {most_common_memory['internal_memory']} GB ({int(most_common_memory['model_count'])} models).")
print("Interpretation: The bar chart compares the number of models available at each memory capacity. The printed value identifies the most frequently represented capacity in this dataset.")

### U4. Distribution of phone weights and minimum–maximum

In [ ]:
q_u4 = """
SELECT `weight` AS phone_weight
FROM cellphones_data
WHERE `weight` IS NOT NULL
"""
with engine.connect() as connection:
    u4 = pd.read_sql(text(q_u4), connection)

display(u4.describe())
plt.hist(u4["phone_weight"], bins=10, edgecolor="black")
plt.title("Distribution of Phone Weights")
plt.xlabel("Weight (grams)")
plt.ylabel("Number of phone models")
plt.show()

print(f"Weight range: {u4.phone_weight.min():.2f} g to {u4.phone_weight.max():.2f} g.")
print(f"Mean weight: {u4.phone_weight.mean():.2f} g; median weight: {u4.phone_weight.median():.2f} g.")
print("Interpretation: The histogram shows the spread of model weights. The minimum and maximum identify the lightest and heaviest phones in the dataset.")

### U5. Battery-size distribution and range

In [ ]:
q_u5 = """
SELECT `battery size` AS battery_size
FROM cellphones_data
WHERE `battery size` IS NOT NULL
"""
with engine.connect() as connection:
    u5 = pd.read_sql(text(q_u5), connection)

display(u5.describe())
plt.hist(u5["battery_size"], bins=10, edgecolor="black")
plt.title("Distribution of Battery Sizes")
plt.xlabel("Battery size (mAh)")
plt.ylabel("Number of phone models")
plt.show()

print(f"Battery-size range: {u5.battery_size.min():,.0f} to {u5.battery_size.max():,.0f} mAh.")
print(f"Mean battery size: {u5.battery_size.mean():,.0f} mAh; median: {u5.battery_size.median():,.0f} mAh.")
print("Interpretation: The histogram summarizes battery capacities across phone models. The range identifies the smallest and largest battery sizes in the dataset.")

# 5B. Bivariate Analysis

### B1. Relationship between user age and price of phones rated

In [ ]:
q_b1 = """
SELECT u.`age` AS user_age, p.`price(INR)` AS phone_price
FROM cellphone_users u
JOIN cellphones_ratings r ON u.`user_id` = r.`user_id`
JOIN cellphones_data p ON r.`cellphone_id` = p.`cellphone_id`
WHERE u.`age` IS NOT NULL AND p.`price(INR)` IS NOT NULL
"""
with engine.connect() as connection:
    b1 = pd.read_sql(text(q_b1), connection)

plt.scatter(b1["user_age"], b1["phone_price"], alpha=0.55)
if b1["user_age"].nunique() > 1:
    slope, intercept = np.polyfit(b1["user_age"], b1["phone_price"], 1)
    x_line = np.linspace(b1["user_age"].min(), b1["user_age"].max(), 100)
    plt.plot(x_line, slope * x_line + intercept)
plt.title("User Age vs Price of Rated Phones")
plt.xlabel("User age")
plt.ylabel("Phone price (INR)")
plt.show()

age_price_corr = b1["user_age"].corr(b1["phone_price"])
print(correlation_summary(age_price_corr))
print("Interpretation: The scatter plot and correlation describe whether older or younger users tend to rate higher-priced phones. A weak correlation means age alone is not a useful predictor of phone price in this dataset. This is an association, not proof of causation.")

### B2. Relationship of main and selfie camera specifications with price

In [ ]:
q_b2 = """
SELECT `price(INR)` AS phone_price,
       `main camera` AS main_camera,
       `selfie camera` AS selfie_camera
FROM cellphones_data
WHERE `price(INR)` IS NOT NULL
"""
with engine.connect() as connection:
    b2 = pd.read_sql(text(q_b2), connection)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(b2["main_camera"], b2["phone_price"])
if b2["main_camera"].nunique() > 1:
    slope, intercept = np.polyfit(b2["main_camera"], b2["phone_price"], 1)
    x_line = np.linspace(b2["main_camera"].min(), b2["main_camera"].max(), 100)
    axes[0].plot(x_line, slope * x_line + intercept)
axes[0].set_title("Main Camera vs Phone Price")
axes[0].set_xlabel("Main camera specification")
axes[0].set_ylabel("Price (INR)")
axes[1].scatter(b2["selfie_camera"], b2["phone_price"])
if b2["selfie_camera"].nunique() > 1:
    slope, intercept = np.polyfit(b2["selfie_camera"], b2["phone_price"], 1)
    x_line = np.linspace(b2["selfie_camera"].min(), b2["selfie_camera"].max(), 100)
    axes[1].plot(x_line, slope * x_line + intercept)
axes[1].set_title("Selfie Camera vs Phone Price")
axes[1].set_xlabel("Selfie camera specification")
axes[1].set_ylabel("Price (INR)")
plt.tight_layout()
plt.show()

main_camera_corr = b2["main_camera"].corr(b2["phone_price"])
selfie_camera_corr = b2["selfie_camera"].corr(b2["phone_price"])
print("Main camera:", correlation_summary(main_camera_corr))
print("Selfie camera:", correlation_summary(selfie_camera_corr))
print("Interpretation: Compare the two correlations to see which camera specification has the stronger linear association with price. These patterns do not establish that camera specifications cause a higher price.")

### B3. Relationship between performance score and phone price

In [ ]:
q_b3 = """
SELECT `performance` AS performance_score, `price(INR)` AS phone_price
FROM cellphones_data
WHERE `performance` IS NOT NULL AND `price(INR)` IS NOT NULL
"""
with engine.connect() as connection:
    b3 = pd.read_sql(text(q_b3), connection)

plt.scatter(b3["performance_score"], b3["phone_price"])
if b3["performance_score"].nunique() > 1:
    slope, intercept = np.polyfit(b3["performance_score"], b3["phone_price"], 1)
    x_line = np.linspace(b3["performance_score"].min(), b3["performance_score"].max(), 100)
    plt.plot(x_line, slope * x_line + intercept)
plt.title("Performance Score vs Phone Price")
plt.xlabel("Performance score")
plt.ylabel("Phone price (INR)")
plt.show()

performance_price_corr = b3["performance_score"].corr(b3["phone_price"])
print(correlation_summary(performance_price_corr))
print("Interpretation: A positive correlation means higher performance scores tend to occur with higher prices; a negative correlation means the reverse. The strength value indicates how closely the two measures move together in this dataset.")

### B4. Most preferred operating system based on user ratings

In [ ]:
q_b4 = """
SELECT p.`operating system` AS operating_system,
       COUNT(r.`rating`) AS rating_count,
       AVG(r.`rating`) AS average_rating
FROM cellphones_data p
JOIN cellphones_ratings r ON p.`cellphone_id` = r.`cellphone_id`
GROUP BY p.`operating system`
ORDER BY average_rating DESC, rating_count DESC
"""
with engine.connect() as connection:
    b4 = pd.read_sql(text(q_b4), connection)

display(b4)
plt.bar(b4["operating_system"].astype(str), b4["average_rating"])
plt.title("Average User Rating by Operating System")
plt.xlabel("Operating system")
plt.ylabel("Average rating")
plt.xticks(rotation=30)
plt.show()

if not b4.empty:
    print("Highest average-rated operating system:", b4.iloc[0]["operating_system"])
if not b4.empty:
    best_os = b4.iloc[0]
    print(f"Highest average rating: {best_os['operating_system']} ({best_os['average_rating']:.2f}; {int(best_os['rating_count'])} rating records).")
print("Interpretation: The operating system at the top has the highest average user rating in this dataset. Rating count provides context, but the highest count is not necessarily the highest average rating.")

### B5. Relationship between release date and phone price

In [ ]:
q_b5 = """
SELECT `release date` AS release_date_text, `price(INR)` AS phone_price
FROM cellphones_data
WHERE `release date` IS NOT NULL AND `price(INR)` IS NOT NULL
"""
with engine.connect() as connection:
    b5 = pd.read_sql(text(q_b5), connection)

b5["release_date"] = pd.to_datetime(b5["release_date_text"], dayfirst=True, errors="coerce")
b5 = b5.dropna(subset=["release_date"])

plt.scatter(b5["release_date"].dt.year, b5["phone_price"])
plt.title("Phone Release Date vs Price")
plt.xlabel("Release year")
plt.ylabel("Phone price (INR)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

release_year_corr = b5["release_date"].dt.year.corr(b5["phone_price"])
print(correlation_summary(release_year_corr))
print("Interpretation: The plot compares model release year with listed price. The correlation summarizes the direction and strength of the linear relationship; it should be interpreted cautiously because the dataset contains only a limited number of phone models.")

# 5C. Multivariate Analysis

### M1. Most preferred brand in each region based on user ratings

In [ ]:
q_m1 = """
WITH region_brand AS (
    SELECT u.`Region(City)` AS region,
           p.`brand` AS brand,
           AVG(r.`rating`) AS average_rating,
           COUNT(r.`rating`) AS rating_count
    FROM cellphone_users u
    JOIN cellphones_ratings r ON u.`user_id` = r.`user_id`
    JOIN cellphones_data p ON r.`cellphone_id` = p.`cellphone_id`
    GROUP BY u.`Region(City)`, p.`brand`
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY region ORDER BY average_rating DESC, rating_count DESC
    ) AS rn
    FROM region_brand
)
SELECT region, brand, average_rating, rating_count
FROM ranked
WHERE rn = 1
ORDER BY region
"""
with engine.connect() as connection:
    m1 = pd.read_sql(text(q_m1), connection)

display(m1)
plt.bar(m1["region"].astype(str), m1["average_rating"])
plt.title("Top-Rated Brand in Each Region")
plt.xlabel("Region")
plt.ylabel("Average rating")
plt.xticks(rotation=45)
for bar, brand in zip(plt.gca().patches, m1["brand"]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(brand), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

print("Interpretation: Each row identifies the brand with the highest average rating in that region. The rating count indicates how many rating records contribute to the comparison. Review this result table to compare regional preferences.")

### M2. Average rated-phone price by occupation; top three occupations

In [ ]:
q_m2 = """
SELECT u.`occupation` AS occupation,
       AVG(p.`price(INR)`) AS average_rated_phone_price,
       COUNT(r.`rating`) AS rating_count
FROM cellphone_users u
JOIN cellphones_ratings r ON u.`user_id` = r.`user_id`
JOIN cellphones_data p ON r.`cellphone_id` = p.`cellphone_id`
GROUP BY u.`occupation`
ORDER BY average_rated_phone_price DESC
"""
with engine.connect() as connection:
    m2 = pd.read_sql(text(q_m2), connection)

display(m2)
top3_occupations = m2.head(3)
plt.bar(top3_occupations["occupation"].astype(str), top3_occupations["average_rated_phone_price"])
plt.title("Top Three Occupations by Average Price of Rated Phones")
plt.xlabel("Occupation")
plt.ylabel("Average rated-phone price (INR)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("Top three occupations:")
display(top3_occupations)
print("Interpretation: The table ranks occupations by the average price of phones rated by users in that occupation. The first three rows are the occupations associated with the highest average rated-phone prices. This is a rating-based proxy, not verified purchase behaviour.")

### M3. Most frequently rated model for each brand, with rating count and price

In [ ]:
q_m3 = """
WITH model_activity AS (
    SELECT p.`brand` AS brand, p.`model` AS model,
           p.`price(INR)` AS phone_price,
           COUNT(r.`rating`) AS rating_count
    FROM cellphones_data p
    JOIN cellphones_ratings r ON p.`cellphone_id` = r.`cellphone_id`
    GROUP BY p.`brand`, p.`model`, p.`price(INR)`
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY brand ORDER BY rating_count DESC, model
    ) AS rn
    FROM model_activity
)
SELECT brand, model, phone_price, rating_count
FROM ranked
WHERE rn = 1
ORDER BY brand
"""
with engine.connect() as connection:
    m3 = pd.read_sql(text(q_m3), connection)

display(m3)
plt.bar(m3["brand"].astype(str), m3["rating_count"])
plt.title("Most Frequently Rated Model per Brand: Rating Count")
plt.xlabel("Brand")
plt.ylabel("Number of rating records")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

print("Interpretation: Each row shows the most frequently rated model within a brand, together with its listed price and rating-record count. The count measures rating activity, not confirmed sales or purchases.")

### M4. Model with the highest average user rating for each brand

In [ ]:
q_m4 = """
WITH model_ratings AS (
    SELECT p.`brand` AS brand, p.`model` AS model,
           AVG(r.`rating`) AS average_rating,
           COUNT(r.`rating`) AS rating_count
    FROM cellphones_data p
    JOIN cellphones_ratings r ON p.`cellphone_id` = r.`cellphone_id`
    GROUP BY p.`brand`, p.`model`
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY brand ORDER BY average_rating DESC, rating_count DESC
    ) AS rn
    FROM model_ratings
)
SELECT brand, model, average_rating, rating_count
FROM ranked
WHERE rn = 1
ORDER BY brand
"""
with engine.connect() as connection:
    m4 = pd.read_sql(text(q_m4), connection)

display(m4)
plt.bar(m4["brand"].astype(str), m4["average_rating"])
plt.title("Highest Average-Rated Model per Brand")
plt.xlabel("Brand")
plt.ylabel("Average user rating")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

print("Interpretation: Each row identifies the highest average-rated model within a brand. Compare the average rating with its rating count; a high average based on fewer ratings should be interpreted with care.")

### M5. Regional top-rated brand and its rated-phone price and user salary

In [ ]:
q_m5 = """
WITH region_brand AS (
    SELECT u.`Region(City)` AS region,
           p.`brand` AS brand,
           AVG(r.`rating`) AS average_rating,
           COUNT(r.`rating`) AS rating_count
    FROM cellphone_users u
    JOIN cellphones_ratings r ON u.`user_id` = r.`user_id`
    JOIN cellphones_data p ON r.`cellphone_id` = p.`cellphone_id`
    GROUP BY u.`Region(City)`, p.`brand`
),
ranked_brands AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY region
               ORDER BY average_rating DESC, rating_count DESC, brand
           ) AS rn
    FROM region_brand
),
regional_top AS (
    SELECT region, brand, average_rating, rating_count
    FROM ranked_brands
    WHERE rn = 1
),
regional_phone_price AS (
    SELECT rt.region, rt.brand,
           AVG(p.`price(INR)`) AS average_rated_phone_price
    FROM regional_top rt
    JOIN cellphone_users u ON u.`Region(City)` = rt.region
    JOIN cellphones_ratings r ON u.`user_id` = r.`user_id`
    JOIN cellphones_data p
      ON r.`cellphone_id` = p.`cellphone_id`
     AND p.`brand` = rt.brand
    GROUP BY rt.region, rt.brand
),
regional_users AS (
    SELECT DISTINCT rt.region, rt.brand,
           u.`user_id`, u.`Salary_in_INR`
    FROM regional_top rt
    JOIN cellphone_users u ON u.`Region(City)` = rt.region
    JOIN cellphones_ratings r ON u.`user_id` = r.`user_id`
    JOIN cellphones_data p
      ON r.`cellphone_id` = p.`cellphone_id`
     AND p.`brand` = rt.brand
),
regional_salary AS (
    SELECT region, brand,
           AVG(`Salary_in_INR`) AS average_user_salary,
           COUNT(*) AS distinct_users_rating_brand
    FROM regional_users
    GROUP BY region, brand
)
SELECT rt.region, rt.brand, rt.average_rating,
       rt.rating_count AS total_brand_rating_count,
       rp.average_rated_phone_price,
       rs.average_user_salary,
       rs.distinct_users_rating_brand
FROM regional_top rt
JOIN regional_phone_price rp
  ON rt.region = rp.region AND rt.brand = rp.brand
JOIN regional_salary rs
  ON rt.region = rs.region AND rt.brand = rs.brand
ORDER BY rt.region
"""
with engine.connect() as connection:
    m5 = pd.read_sql(text(q_m5), connection)

display(m5)

if not m5.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(m5["region"].astype(str), m5["average_rated_phone_price"])
    axes[0].set_title("Average Price of Rated Phones from Regional Top Brand")
    axes[0].set_xlabel("Region")
    axes[0].set_ylabel("Average phone price (INR)")
    axes[0].tick_params(axis="x", rotation=35)

    axes[1].bar(m5["region"].astype(str), m5["average_user_salary"])
    axes[1].set_title("Average Salary of Distinct Users Rating Regional Top Brand")
    axes[1].set_xlabel("Region")
    axes[1].set_ylabel("Average salary (INR)")
    axes[1].tick_params(axis="x", rotation=35)

    plt.tight_layout()
    plt.show()

print(
    "Interpretation: The query first identifies the highest-average-rated brand in "
    "each region. It then calculates the average price of rated phones from that brand "
    "and the average salary of distinct users who rated that brand. Counting each user "
    "once for salary avoids over-weighting users who submitted more ratings. These are "
    "rating-based patterns, not verified purchase behaviour."
)

## 6. Overall findings and limitations

### Evidence-based summary

The tables and charts generated in Sections 5A–5C are the evidence for the findings below. The next code cell prints a concise, data-driven summary directly from the SQL result sets created above. Use those actual outputs when discussing the project; do not substitute assumptions or market-wide claims.

### Limitations

1. The ratings dataset contains user ratings, not verified transaction records. Rating counts are used only as a proxy for rating activity.
2. Correlation and visual association do not establish causation.
3. The dataset contains 33 phone models, 99 user profiles and 990 rating records; findings describe this supplied sample and may not represent the entire smartphone market.
4. Possible outliers are reported for inspection only. No outlier treatment is performed, as required by the assignment.
5. Regional or brand rankings are based on average ratings; rating counts are shown as supporting context. Small sample sizes can affect rankings.


In [ ]:
print("SAAP Stage 1 — data-driven findings")
print("- Phone-price range (INR):", f"{u1['phone_price'].min():,.2f} to {u1['phone_price'].max():,.2f}")
print("- User-salary range (INR):", f"{u2['salary'].min():,.2f} to {u2['salary'].max():,.2f}")
print("- Most common internal-memory capacity (GB):", most_common_memory['internal_memory'])
print("- Lightest to heaviest phone (g):", f"{u4['phone_weight'].min():.0f} to {u4['phone_weight'].max():.0f}")
print("- Battery-size range (mAh):", f"{u5['battery_size'].min():,.0f} to {u5['battery_size'].max():,.0f}")
if not b4.empty:
    print("- Highest average-rated operating system:", b4.iloc[0]['operating_system'])
if not m1.empty:
    print("- Regional top-rated brands (region → brand):")
    for _, row in m1.iterrows():
        print(f"  {row['region']} → {row['brand']} (average rating {row['average_rating']:.2f}; n={int(row['rating_count'])})")
if not m2.empty:
    print("- Top 3 occupations by average price of rated phones:", ", ".join(top3_occupations['occupation'].astype(str).tolist()))
print("\nInterpretation: These values summarize this dataset only. Rating activity is not verified purchasing, and observed relationships are not causal.")


## 8. Final notebook execution check

This final cell confirms that all 15 analysis result variables were created during the current kernel run. It does not replace the requirement to run every cell from top to bottom.

In [ ]:
required_analysis_results = [
    "u1", "u2", "u3", "u4", "u5",
    "b1", "b2", "b3", "b4", "b5",
    "m1", "m2", "m3", "m4", "m5"
]

missing_results = [
    variable_name for variable_name in required_analysis_results
    if variable_name not in globals()
]

if missing_results:
    raise RuntimeError(
        "Some analysis result variables are missing: " + ", ".join(missing_results) +
        ". Run the notebook from the first cell to the last cell."
    )

print("All 15 SQL-based analysis result variables were created in this run.")
print("Before submission, use Kernel -> Restart & Run All and save the executed .ipynb file.")